In [9]:
from difflib import SequenceMatcher
import numpy as np
import os
import subprocess
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT = os.environ["PROJECT_ROOT"]

In [10]:
"""
align_ocr.py
────────────
Robust alignment of noisy OCR text to a clean reference text.

Strategy
--------
1.  Normalise both texts: lower-case, strip diacritics, collapse whitespace,
    remove hyphens that break words across lines.
2.  Tokenise both into word lists.
3.  Use a *sliding-anchor* approach:
      • For each OCR image, extract its word list.
      • Search the reference word-list for the best matching window using
        a fast character-level n-gram overlap (much faster than full DP for
        large texts, yet robust to OCR substitutions).
      • Once the anchor is found, run a lightweight word-level DP alignment
        within that window to map every OCR word → the closest reference word.
4.  Reconstruct the reference lines that correspond to each OCR line.

Drift-recovery additions
------------------------
A.  Per-page restart: each page\'s search baseline is derived from its
    position in the OCR sequence (cumulative OCR words so far × the
    OCR-to-reference ratio), not just the previous cursor. This makes
    every page roughly independent and stops one bad anchor from
    cascading into the next.
B.  Backward overlap scales with the current OCR page length, so a few
    pages of small forward drift do not put the true location out of reach
    of the next search.
C.  Forward search distance is capped at `forward_mult * len(ocr_page)`,
    so a spurious n-gram match deep in a repetitive reference cannot
    hijack the cursor.
D.  If the best n-gram score is below `min_anchor_score` (very weak match
    overall) the anchor falls back to `search_from` rather than trusting
    the marginal winner.
"""

import re
import unicodedata
from pathlib import Path
from difflib import SequenceMatcher


# ──────────────────────────────────────────────
#  1.  Text normalisation helpers
# ──────────────────────────────────────────────

def strip_diacritics(text: str) -> str:
    """Remove combining diacritical marks (keeps base letters)."""
    return "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )


def normalise(text: str) -> str:
    """
    Lower-case, strip diacritics, join hyphenated line-breaks,
    collapse whitespace.
    """
    text = re.sub(r"-\s*\n\s*", "", text)
    text = text.lower()
    text = strip_diacritics(text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenise(text: str) -> list[str]:
    return normalise(text).split()


# ──────────────────────────────────────────────
#  2.  Fast n-gram anchor search
# ──────────────────────────────────────────────

def char_ngrams(text: str, n: int = 3) -> set[str]:
    return {text[i:i+n] for i in range(len(text) - n + 1)}


def ngram_similarity(a: str, b: str, n: int = 3) -> float:
    """Jaccard similarity on character n-grams."""
    ga, gb = char_ngrams(a, n), char_ngrams(b, n)
    if not ga or not gb:
        return 0.0
    return len(ga & gb) / len(ga | gb)


def find_anchor(
    ocr_words: list[str],
    ref_words: list[str],
    search_from: int = 0,
    probe_len: int = 30,
    step: int = 10,
    window_mult: int = 6,
    forward_mult: int = 3,
    min_anchor_score: float = 0.10,
) -> tuple[int, float]:
    """
    Find the index in ref_words where ocr_words most likely begins.

    The search is bounded to
    ``[search_from, search_from + forward_mult * len(ocr_words)]``
    so a spurious match far down the reference cannot hijack the cursor.
    If the best score is below ``min_anchor_score`` the function returns
    ``(search_from, score)`` — i.e. trust the cursor rather than the
    marginal winner.

    Returns
    -------
    (best_idx, best_score)
    """
    probe = " ".join(ocr_words[:probe_len])
    window_size = probe_len * window_mult

    best_idx = search_from
    best_score = -1.0

    max_end = len(ref_words) - probe_len
    bounded_end = min(max_end, search_from + forward_mult * len(ocr_words))

    for i in range(search_from, bounded_end, step):
        window = " ".join(ref_words[i: i + window_size])
        score = ngram_similarity(probe, window)
        if score > best_score:
            best_score = score
            best_idx = i

    if best_score < min_anchor_score:
        return search_from, best_score

    return best_idx, best_score


# ──────────────────────────────────────────────
#  3.  Word-level DP alignment (Needleman-Wunsch lite)
# ──────────────────────────────────────────────

def word_similarity(a: str, b: str) -> float:
    """Character-level similarity between two (normalised) words."""
    if a == b:
        return 1.0
    return SequenceMatcher(None, a, b).ratio()


def align_word_sequences(
    ocr_words: list[str],
    ref_words: list[str],
    match_reward: float = 2.0,
    mismatch_penalty: float = -1.0,
    gap_penalty: float = -0.5,
    sim_threshold: float = 0.6,
) -> list[tuple[int | None, int | None]]:
    """
    Semi-global DP alignment: OCR words vs reference words.
    Returns list of (ocr_idx, ref_idx) pairs (None = gap).
    """
    m, n = len(ocr_words), len(ref_words)

    dp = [[0.0] * (n + 1) for _ in range(m + 1)]
    tb = [[(0, 0)] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        dp[i][0] = dp[i-1][0] + gap_penalty
        tb[i][0] = (i-1, 0)

    for j in range(1, n + 1):
        dp[0][j] = 0.0
        tb[0][j] = (0, j-1)

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            sim = word_similarity(ocr_words[i-1], ref_words[j-1])
            score = match_reward if sim >= sim_threshold else mismatch_penalty * (1 - sim)

            diag = dp[i-1][j-1] + score
            up   = dp[i-1][j]   + gap_penalty
            left = dp[i][j-1]   + gap_penalty

            best = max(diag, up, left)
            dp[i][j] = best
            if best == diag:
                tb[i][j] = (i-1, j-1)
            elif best == up:
                tb[i][j] = (i-1, j)
            else:
                tb[i][j] = (i, j-1)

    last_row = dp[m]
    j_end = max(range(n + 1), key=lambda j: last_row[j])

    pairs: list[tuple[int | None, int | None]] = []
    i, j = m, j_end
    while i > 0 or j > 0:
        pi, pj = tb[i][j]
        if pi == i - 1 and pj == j - 1:
            pairs.append((i - 1, j - 1))
        elif pi == i - 1:
            pairs.append((i - 1, None))
        else:
            pairs.append((None, j - 1))
        i, j = pi, pj

    pairs.reverse()
    return pairs


# ──────────────────────────────────────────────
#  4.  Main alignment driver
# ──────────────────────────────────────────────

def align_ocr_to_reference(
    full_ref_text: str,
    ocr_dir: Path,
    *,
    probe_len: int = 25,
    ref_window_words: int = 300,
    overlap_words: int = 1500,
    forward_mult: int = 3,
    min_anchor_score: float = 0.10,
    per_page_restart: bool = True,
) -> list[tuple[str, list[str]]]:
    """
    For every OCR file in ocr_dir (sorted naturally), find its position
    in the reference and return aligned lines.

    Parameters
    ----------
    overlap_words : how far BEHIND the search baseline the anchor search
        may look. Default 1500 (~one manuscript page worth of words).
    forward_mult : the anchor search end is capped at
        ``search_from + forward_mult * len(ocr_words)``. Default 3.
    min_anchor_score : if the best window\'s n-gram score is below this,
        treat the match as unreliable and fall back to ``search_from``.
    per_page_restart : if True (default), the search baseline for each
        page is the *earlier* of (a) the running ref cursor and (b) the
        cumulative-OCR-words estimate (``cumulative_ocr * ref/ocr ratio``).
        This makes each page anchor near where its position in the OCR
        sequence says it should be, rather than purely inheriting drift
        from the previous page.
    """
    ref_words_raw = full_ref_text.split()
    ref_words_norm = tokenise(full_ref_text)

    ocr_files = sorted(
        ocr_dir.glob("*.txt"),
        key=lambda p: [int(c) if c.isdigit() else c.lower()
                       for c in re.split(r"(\d+)", p.name)],
    )

    # ── Pre-pass: count total OCR words for the per-page restart ratio ──
    # We use the raw whitespace split (not normalised) here because it\'s
    # cheap and the difference vs. normalised is small enough that it
    # doesn\'t materially change the per-page estimate (the anchor search
    # itself still refines using the normalised words).
    if per_page_restart:
        total_raw_ocr = sum(
            len(p.read_text(encoding="utf-8").split()) for p in ocr_files
        )
        ref_per_raw_ocr = (
            len(ref_words_norm) / total_raw_ocr if total_raw_ocr else 0.0
        )
    else:
        ref_per_raw_ocr = 0.0

    aligned_doc: list[tuple[str, list[str]]] = []
    ref_cursor = 0
    cumulative_raw_ocr = 0

    for ocr_path in ocr_files:
        raw_ocr = ocr_path.read_text(encoding="utf-8")
        raw_word_count = len(raw_ocr.split())

        ocr_lines_raw = raw_ocr.splitlines()
        ocr_words_norm: list[str] = []
        line_word_counts: list[int] = []

        for line in ocr_lines_raw:
            words = tokenise(line)
            line_word_counts.append(len(words))
            ocr_words_norm.extend(words)

        if not ocr_words_norm:
            aligned_doc.append((ocr_path.stem, []))
            cumulative_raw_ocr += raw_word_count
            continue

        # ── Per-page restart: where in the reference SHOULD this page be? ──
        # Estimated from the cumulative OCR position before this page.
        if per_page_restart:
            expected_anchor = int(cumulative_raw_ocr * ref_per_raw_ocr)
        else:
            expected_anchor = ref_cursor

        # Use the earlier of cursor and expected as the search baseline:
        # • cursor   — accurate when the previous page aligned well
        # • expected — robust when the previous page drifted or failed
        search_baseline = min(ref_cursor, expected_anchor)

        # Backward overlap scales with OCR page length so accumulated
        # forward drift can be recovered even when it exceeds the default.
        backward_window = max(overlap_words, len(ocr_words_norm))
        anchor_idx, anchor_score = find_anchor(
            ocr_words_norm,
            ref_words_norm,
            search_from=max(0, search_baseline - backward_window),
            probe_len=min(probe_len, len(ocr_words_norm)),
            forward_mult=forward_mult,
            min_anchor_score=min_anchor_score,
        )

        ref_end = min(len(ref_words_norm), anchor_idx + len(ocr_words_norm) + ref_window_words)
        ref_window_norm = ref_words_norm[anchor_idx:ref_end]
        ref_window_raw  = ref_words_raw[anchor_idx:ref_end]

        pairs = align_word_sequences(ocr_words_norm, ref_window_norm)

        ocr_to_ref: dict[int, int] = {}
        for ocr_i, ref_i in pairs:
            if ocr_i is not None and ref_i is not None:
                ocr_to_ref[ocr_i] = ref_i

        aligned_lines: list[str] = []
        ocr_word_cursor = 0

        for word_count in line_word_counts:
            if word_count == 0:
                aligned_lines.append("")
                continue

            ref_indices = [
                ocr_to_ref[ocr_word_cursor + k]
                for k in range(word_count)
                if (ocr_word_cursor + k) in ocr_to_ref
            ]

            if ref_indices:
                lo, hi = min(ref_indices), max(ref_indices)
                aligned_lines.append(" ".join(ref_window_raw[lo: hi + 1]))
            else:
                aligned_lines.append("")

            ocr_word_cursor += word_count

        if ocr_to_ref:
            last_ref_in_window = max(ocr_to_ref.values())
            ref_cursor = anchor_idx + last_ref_in_window + 1
        else:
            ref_cursor = anchor_idx + len(ocr_words_norm)

        aligned_doc.append((ocr_path.stem, aligned_lines))
        exp_str = f", expected≈{expected_anchor}" if per_page_restart else ""
        print(f"✓ {ocr_path.stem}: anchor={anchor_idx} "
              f"(score={anchor_score:.3f}{exp_str}), "
              f"lines={len(aligned_lines)}, "
              f"matched={len(ocr_to_ref)}/{len(ocr_words_norm)} words")

        cumulative_raw_ocr += raw_word_count

    return aligned_doc


# ──────────────────────────────────────────────
#  5.  Save helpers
# ──────────────────────────────────────────────

def save_aligned(
    aligned_doc: list[tuple[str, list[str]]],
    out_path: Path,
    side_by_side: bool = True,
) -> None:
    """Save aligned reference lines per image."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for img_name, lines in aligned_doc:
            f.write(f"\n{'='*10} IMAGE: {img_name} {'='*10}\n")
            for line in lines:
                f.write(line + "\n")
    print(f"\n💾 Saved to: {out_path}")


def save_side_by_side(
    ocr_dir: Path,
    aligned_doc: list[tuple[str, list[str]]],
    out_path: Path,
    col_width: int = 55,
) -> None:
    """Two-column view: original OCR line | aligned reference line."""
    ocr_map = {p.stem: p for p in ocr_dir.glob("*.txt")}
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, "w", encoding="utf-8") as f:
        header = f"{'OCR':<{col_width}}  {'ALIGNED REF'}"
        f.write(header + "\n" + "─" * (col_width * 2 + 4) + "\n")

        for img_name, ref_lines in aligned_doc:
            f.write(f"\n{'='*10} {img_name} {'='*10}\n")
            ocr_lines = (ocr_map[img_name].read_text(encoding="utf-8").splitlines()
                         if img_name in ocr_map else [""] * len(ref_lines))
            for ocr_line, ref_line in zip(ocr_lines, ref_lines):
                ocr_trunc = ocr_line[:col_width].ljust(col_width)
                f.write(f"{ocr_trunc}  {ref_line}\n")

    print(f"📊 Side-by-side saved to: {out_path}")


In [ ]:
or_trans = Path(PROJECT_ROOT, "data/raw/AlbucE.txt")
ocr_dir  = Path(PROJECT_ROOT, "data/processed/transcription/ocr_kept_20260515_104644")

with open(or_trans, "r", encoding="utf-8") as f:
    ref_text = f.read()

aligned_doc = align_ocr_to_reference(ref_text, ocr_dir)

# Clean aligned output
save_aligned(
    aligned_doc,
    Path(PROJECT_ROOT, "tests/ocr/AlbucE_aligned_anchored6.txt"),
)

# Optional: two-column review file
save_side_by_side(
    ocr_dir,
    aligned_doc,
    Path(PROJECT_ROOT, "tests/ocr/AlbucE_aligned_sidebyside.txt"),
)

✓ 05_garde_001_full: anchor=0 (score=0.239, expected≈0), lines=100, matched=656/683 words
✓ 06_f_001v_002_full: anchor=740 (score=0.205, expected≈750), lines=202, matched=1280/1367 words
✓ 07_f_002v_003_full: anchor=2108 (score=0.214, expected≈2237), lines=192, matched=1242/1326 words
✓ 08_f_003v_004_full: anchor=3513 (score=0.250, expected≈3683), lines=194, matched=1162/1287 words
✓ 09_f_004v_005_full: anchor=4916 (score=0.220, expected≈5082), lines=191, matched=1245/1322 words
✓ 10_f_005v_006_full: anchor=6420 (score=0.238, expected≈6514), lines=187, matched=1163/1238 words
✓ 11_f_006v_007_full: anchor=7788 (score=0.229, expected≈7859), lines=196, matched=1141/1224 words
✓ 12_f_007v_008_full: anchor=9097 (score=0.310, expected≈9188), lines=201, matched=1211/1278 words
✓ 13_f_008v_009_full: anchor=10462 (score=0.269, expected≈10576), lines=197, matched=1210/1278 words
✓ 14_f_009v_010_full: anchor=11830 (score=0.257, expected≈11972), lines=193, matched=1183/1256 words
✓ 15_f_010v_011_f